## ME 481: Whole Body Biomechanics - Cost of Transport Lab

In this lab, you will explore how efficiently the human body uses energy during walking and running at different speeds. By analyzing metabolic data, you will calculate the Cost of Transport (CoT) and test theoretical predictions from biomechanical models, including the inverted pendulum model of walking and the Froude number framework for gait transitions.

**How to use this notebook:**
- Read each section's short background.
- Complete and run the code cells. **Note: You will be asked to complete some of the code yourself! These portions are denoted by #TODO**
- Answer the discussion questions.
- Submit this completed notebook (**as HTML and .ipynb**) to Canvas.

**Course Context:**
This lab directly applies concepts from our course textbook on dynamic models of human locomotion. You will use the Froude number to predict optimal walking speeds and gait transitions, then verify these predictions using real metabolic data.

## Lab Overview and Learning Objectives

### What is This Lab About?

This lab uses physiological measurements and biomechanical theory to investigate human locomotion efficiency. You will observe a subject walking and running at different speeds on a treadmill while wearing a portable VO₂ analyzer. You will use Python in Google Colab to:

1. Convert raw oxygen consumption (VO₂) into metabolic power
2. Calculate Net Cost of Transport (energy per unit mass per unit distance)
3. Normalize speeds using the dimensionless Froude number
4. Compare your results to classic biomechanical theories and Wilson's transport efficiency chart

### By the end of the lab, you will be able to:

1. **Predict optimal locomotion speeds and gait transitions** using the Froude number and inverted pendulum model
2. **Convert metabolic measurements** from VO₂ (mL/min) to mechanical power (Watts) and Cost of Transport (J·kg⁻¹·m⁻¹)
3. **Validate biomechanical models** by comparing predicted vs. measured optimal walking speeds and gait transition points

## Data Used in this lab

The data for this lab was collected using the following protocol:

**Subject Preparation:**
- The subject's mass ($m$, in kg) and leg length ($l$, in meters from greater trochanter to floor) were measured
- The subject was fitted with a VO₂ Master portable metabolic analyzer to measure oxygen consumption in real-time

**Baseline Measurement:**
- A 3-minute resting baseline ($\dot{V}_{O_{2,rest}}$) was collected while the subject stood quietly on the treadmill

**Speed Sweep Protocol:**
The subject completed eight continuous 3-minute stages on a treadmill at target speeds:
1. **Walk 1.5 mph**
2. **Walk 2.5 mph**
3. **Walk 3.0 mph**
4. **Walk 3.5 mph**
5. **Walk 5.0 mph**
6. **Walk 6.0 mph**
7. **Run 6.0 mph**
8. **Run 7.0 mph**

**Data Export:**
The VO₂ Master app exports continuous data for all active stages. These raw VO₂ values (in mL/min) are provided on Box. The code cells below will download the data to the current directory for you.

## I. Background

### Theoretical Foundation: The Inverted Pendulum Model

Human walking operates mechanically like an inverted pendulum. During each step, the body "vaults" over the stance leg, converting gravitational potential energy into kinetic energy and back again. This passive exchange minimizes the muscular work needed to maintain forward motion.

However, this mechanism is only efficient within a specific range of speeds. Walk too slowly, and you waste energy stabilizing your body. Walk too fast, and the pendulum mechanics break down—you can no longer keep your stance leg relatively straight, and the gait becomes mechanically unstable.

### The Froude Number: Predicting Optimal Speed

The **Froude number** is a dimensionless parameter that normalizes walking speed by the pendulum-like dynamics of the leg:

$$Fr = \frac{v^2}{gl}$$

where:
- $v$ = walking velocity (m/s)
- $g$ = gravitational acceleration (9.81 m/s²)
- $l$ = leg length (m)

**Key Theoretical Predictions:**
- **Optimal walking efficiency** occurs around $Fr \approx 0.25$ (lowest cost of transport)
- **Walk-run transition** occurs at $Fr \approx 0.5$ (where running begins to lower cost of transport)
- **Maximum walking speed** occurs at $Fr=1.0$ (where inverted pendulum begins to leave the ground)

Because the Froude number accounts for leg length, it allows for comparisons between individuals of different sizes (or even across species).

### Cost of Transport: Energy per Distance

The **Cost of Transport (CoT)** in this context quantifies how much metabolic energy it takes to move a unit of body mass over a unit of distance:

$$\text{Net CoT} = \frac{\dot{P}_{met} - \dot{P}_{rest}}{m v}$$

where:
- $\dot{P}_{met}$ = metabolic power during activity (W)
- $\dot{P}_{rest}$ = resting metabolic power (W)
- $m$ = body mass (kg)
- $v$ = velocity (m/s)

Units: **J·kg⁻¹·m⁻¹** (or equivalently, cal·g⁻¹·km⁻¹)

### From Oxygen to Energy

We measure metabolism using oxygen consumption ($\dot{VO}_2$, typically in mL/min). To convert this to mechanical power, we use the **energy equivalent of oxygen**:

$$\dot{P}_{met} = \dot{VO}_2 \times 20.1 \, \text{J/mL}$$

**Why 20.1 J/mL?**

- Oxygen consumption itself is not energy; it is a proxy for how much chemical energy is being burned in the mitochondria.
- The **energy equivalent of oxygen** links VO₂ to energy expenditure based on the mix of fuels being metabolized.
- The mix of fuels is quantified by the **respiratory exchange ratio (RER)**, defined as:

  $$\text{RER} = \frac{\dot{V}_{CO_2}}{\dot{V}_{O_2}}$$

  - RER ≈ 0.7 for primarily fat metabolism
  - RER ≈ 1.0 for primarily carbohydrate metabolism
  - RER ≈ 0.85 for a typical mixed diet (common during steady-state walking/running)

- A mixed fuel metabolism (RER ≈ 0.85) yields an energy equivalent around **20.1 J/mL**, which is why we use it in this lab.

### Steady-State Physiology

When you start exercising, your oxygen consumption doesn't instantly jump to the required level—it follows a **first-order system response**, gradually rising before plateauing. This is why each stage in the protocol lasts 3 minutes: we need the final portion of data to represent true steady-state metabolism at that speed.

### Transport Efficiency

In 1973, S.S. Wilson published an article in Scientific American comparing transport efficiency across biological and mechanical systems. His chart (expressed in cal·g⁻¹·km⁻¹) showed that:
- Humans walking: ~0.75 cal·g⁻¹·km⁻¹
- Humans on bicycles: ~0.15 cal·g⁻¹·km⁻¹
- Fish, birds, horses, cars, and helicopters all fall at different points on the spectrum

<img src="https://static.scientificamerican.com/dam/m/715ae217b6961ab5/original/saw1125Gsci31_d_DEFAULT.gif?m=1759758107.705&w=1300" alt="Comparative transport efficiency across modes" width="700"/>

*Source: [Scientific American — “A Human on a Bicycle Is Among the Most Efficient Forms of Travel in the Animal Kingdom”](https://www.scientificamerican.com/article/a-human-on-a-bicycle-is-among-the-most-efficient-forms-of-travel-in-the/)*

## II. Analysis

### 1) Subject and Theoretical Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Subject-specific parameters
mass_kg = 75
leg_length_m = 1.0033

# Theoretical Froude numbers for different gait stages
fr_opt_theory = 0.25
fr_transition_theory = 0.50
fr_maximum_theory = 1.0

g = 9.81  # Acceleration due to gravity in m/s^2

### 2) Predict Speeds from Froude Number

Use the subject leg length to compute the predicted speeds for:
- Maximum walking efficiency at $Fr \approx 0.25$
- Mechanical walk-run transition at $Fr \approx 0.5$
- Maximum walking speed where the inverted pendulum model is valid

In [87]:
# TODO: Calculate predicted speeds in m/s using the Froude number formula
v_opt_pred_mps = #TODO
v_transition_pred_mps = #TODO
v_maximum_pred_mps = #TODO

# Convert predicted speeds from m/s to mph
mps_to_mph = 2.23694
v_opt_pred_mph = v_opt_pred_mps * mps_to_mph
v_transition_pred_mph = v_transition_pred_mps * mps_to_mph
v_maximum_pred_mph = v_maximum_pred_mps * mps_to_mph

print(
    f"Predicted optimal walking speed (Fr=0.25): "
    f"{v_opt_pred_mps:.3f} m/s ({v_opt_pred_mph:.2f} mph)"
)
print(
    f"Predicted walk-run transition speed (Fr=0.50): "
    f"{v_transition_pred_mps:.3f} m/s ({v_transition_pred_mph:.2f} mph)"
)
print(
    f"Predicted maximum walking speed (Fr=1.0): "
    f"{v_maximum_pred_mps:.3f} m/s ({v_maximum_pred_mph:.2f} mph)"
)

### 3) Load, Clean, and Organize Metabolic Data

In [ ]:
import requests
import os


def download_file(url, save_path):
    """
    Downloads a file from a given URL and saves it to a specified path.

    Args:
        url (str): The URL of the file to download.
        save_path (str): The local path where the file will be saved.

    Returns:
        str: The path where the file was saved.

    Raises:
        requests.exceptions.RequestException: If an error occurs during the download.
    """
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(save_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    return save_path


# Download URLs
data_url = 'https://uofi.box.com/shared/static/qphvu6s4hobd0vnb2ycbx8awsuppdjs2.csv'

# Save paths (local)
metabolic_data_path = 'metabolic_data.csv'

# Download files
download_file(data_url, metabolic_data_path)

if os.path.exists(metabolic_data_path):
    print("Data files downloaded successfully!")

In [ ]:
vo2_data = pd.read_csv(metabolic_data_path)

vo2_data.head()

Our $\dot{V}_{O{_2}}$ analyzer does per-breath analysis. This means that sometimes it will reject a breath if it can't cleanly detect it. When this happens, it outputs a value of zero. So, we will gap fill any zeros in our $\dot{V}_{O{_2}}$ data by interpolation.

In [ ]:
# Interpolate zero values in all VO2-related columns
vo2_cols = [col for col in vo2_data.columns if "VO2" in col]

zero_counts_before = (vo2_data[vo2_cols] == 0).sum()

vo2_data[vo2_cols] = (
    vo2_data[vo2_cols]
    .replace(0, np.nan)
    .interpolate(method="linear", limit_direction="both")
)

zero_counts_after = (vo2_data[vo2_cols] == 0).sum()

print("VO2 zero counts before interpolation:")
print(zero_counts_before)
print("\nVO2 zero counts after interpolation:")
print(zero_counts_after)

$\dot{V}_{O{_2}}$ data is also noisy because, instead of obtaining values at a constant sampling rate, it is getting values per-breath. The analyzer we have automatically interpolates so that there is a constant sampling rate in the output files. This results in step-wise artifacts in the data. We will use a 30-second moving average filter to smooth out the data.

In [ ]:
# Apply 30-second moving average filter to VO2 signals
# (assumes ~1 Hz data; computes window from Time[s] to be safe)

dt = vo2_data["Time[s]"].diff().median()
window_samples = max(1, int(round(30 / dt)))

# Keep raw columns, then overwrite with smoothed versions
vo2_data["VO2_raw[mL/min]"] = vo2_data["VO2[mL/min]"]
vo2_data["VO2_raw[mL/kg/min]"] = vo2_data["VO2[mL/kg/min]"]

vo2_data["VO2[mL/min]"] = (
    vo2_data["VO2_raw[mL/min]"]
    .rolling(window=window_samples, min_periods=1, center=True)
    .mean()
)

vo2_data["VO2[mL/kg/min]"] = (
    vo2_data["VO2_raw[mL/kg/min]"]
    .rolling(window=window_samples, min_periods=1, center=True)
    .mean()
)

Now let's plot our cleaned $\dot{V}_{O{_2}}$ data over time to inspect it visually.

In [ ]:
# TODO: Plot VO2 (mL/min) against time

Here, we will separate our continuous data into our protocol stages based on time.

In [ ]:
# Define stage protocol for this dataset
stage_labels = [
    "Walk 1.5 mph",
    "Walk 2.5 mph",
    "Walk 3.0 mph",
    "Walk 3.5 mph",
    "Walk 5.0 mph",
    "Walk 6.0 mph",
    "Run 6.0 mph",
    "Run 7.0 mph",
]
stage_duration_s = 180

# Assign stage windows from the start of the recording
recording_duration_s = vo2_data["Time[s]"].max(
) - vo2_data["Time[s]"].min() + 1
expected_with_rest = (len(stage_labels) + 1) * stage_duration_s
expected_no_rest = len(stage_labels) * stage_duration_s
include_rest = abs(recording_duration_s -
                   expected_with_rest) <= abs(recording_duration_s - expected_no_rest)

vo2_data["Stage"] = pd.NA
start_time = vo2_data["Time[s]"].min()

for index, stage_name in enumerate(stage_labels):
    window_start = start_time + index * stage_duration_s
    window_end = window_start + stage_duration_s
    vo2_data.loc[
        (vo2_data["Time[s]"] >= window_start) & (
            vo2_data["Time[s]"] < window_end),
        "Stage",
    ] = stage_name

print(f"Assigned stages: {', '.join(stage_labels)}")
print(f"Stage duration used: {stage_duration_s} s")

Now, we will compute the average $\dot{V}_{O{_2}}$ for each stage by averaging the values.

In [ ]:
steady_state_vo2 = {}

for stage in stage_labels:
    stage_data = vo2_data[vo2_data["Stage"] == stage]
    # TODO: Average the VO2 values for this stage and store in steady_state_vo2

print("Average VO2 over each full stage:")
for stage, vo2 in steady_state_vo2.items():
    print(f"{stage}: {vo2:.2f} mL/min")

This cell organizes all stage-level inputs needed for the remaining analysis:

- Defines treadmill speeds for each protocol stage in **mph** and converts to **m/s**
- Sets the subject’s **resting VO₂** in mL/min using body-mass-normalized resting VO₂
- Builds an ordered NumPy array of **stage-average VO₂** values from `steady_state_vo2`

In [ ]:
# Stage speeds in mph and m/s (same order as stage_labels)
stage_speeds_mph = np.array(
    [1.5, 2.5, 3.0, 3.5, 5.0, 6.0, 6.0, 7.0], dtype=float)
speeds_mps = stage_speeds_mph / mps_to_mph

# Resting VO2 (mL/min)
# We collected a resting metabolic rate trial separately. The subject's resting VO2 is approximately 4.8 mL/kg/min, so we can compute the resting VO2 in mL/min by multiplying by body mass.
vo2_rest_ml_min = 4.8 * mass_kg

# Stage VO2 values (mL/min) in the same order as stage_labels
vo2_stage_ml_min = np.array([steady_state_vo2[label]
                            for label in stage_labels], dtype=float)

### 3) Convert VO₂ to Metabolic Power (W)

Use the oxygen energy equivalent of $20.1 \ \text{J/mL}$ and convert from minutes to seconds.

In [ ]:
oxygen_equivalent_j_per_ml = 20.1

# TODO: Convert VO2 (mL/min) to power (W = J/s)
p_rest_w = #TODO: Reference the equation in the background section to compute power for each stage using vo2_rest_ml_min
p_stage_w = #TODO: Reference the equation in the background section to compute power for each stage using vo2_stage_ml_min

print(f"Resting metabolic power: {p_rest_w:.2f} W")
for label, power in zip(stage_labels, p_stage_w):
    print(f"{label:>10}: {power:.2f} W")

### 4) Compute Net Cost of Transport (J·kg⁻¹·m⁻¹)

Use:

$$\text{Net CoT} = \frac{\dot{P}_{met} - \dot{P}_{rest}}{m\,v}$$

for each of the eight movement stages.

In [ ]:
net_cot_j_per_kg_m = # TODO: Compute net CoT for each stage

for label, cot in zip(stage_labels, net_cot_j_per_kg_m):
    print(f"{label:>10}: {cot:.4f} J·kg⁻¹·m⁻¹")

### 5) Convert Speeds to Froude Number

Use:

$$Fr = \frac{v^2}{g\,l}$$

for each stage speed to compare with textbook predictions.

In [ ]:
fr_stage = # TODO: Compute stage-level Froude numbers using the stage speeds and leg length

for label, fr in zip(stage_labels, fr_stage):
    print(f"{label:>10}: Fr = {fr:.3f}")

### 6) Convert Net CoT Units (cal·g⁻¹·km⁻¹)

Use the conversion:

$$1\ \text{cal} = 4.184\ \text{J}$$

to convert to the units used in the Scientific American article:

$$
\text{Wilson Efficiency} = \frac{\text{Net CoT}\;(\text{J}\cdot\text{kg}^{-1}\cdot\text{m}^{-1})}{4.184}
$$

In [ ]:
wilson_eff_cal_g_km = # TODO: Convert net cost of transport to Wilson units

for label, val in zip(stage_labels, wilson_eff_cal_g_km):
    print(f"{label:>10}: {val:.3f} cal·g⁻¹·km⁻¹")

### 7) Summarize Results in a Table

In [ ]:
results_df = pd.DataFrame({
    "Stage": stage_labels,
    "Speed (m/s)": speeds_mps,
    "VO2 (mL/min)": vo2_stage_ml_min,
    "Metabolic Power (W)": p_stage_w,
    "Net CoT (J·kg^-1·m^-1)": net_cot_j_per_kg_m,
    "Froude Number": fr_stage,
    "Wilson Efficiency (cal·g^-1·km^-1)": wilson_eff_cal_g_km
})

results_df

### 8) Generate Plots

Here, we have provided the plotting code, but you will need to fit curves to the experimental data. Details can be found after the #TODO tags.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(17, 4.5))

# Split walking (first 6) and running (last 2) points
walk_count = 6
x_speed_walk = speeds_mps[:walk_count]
x_fr_walk = fr_stage[:walk_count]
y_walk = net_cot_j_per_kg_m[:walk_count]

x_speed_run = speeds_mps[walk_count:]
x_fr_run = fr_stage[walk_count:]
y_run = net_cot_j_per_kg_m[walk_count:]

# Quadratic fits to walking points
coef_speed_walk = #TODO: Fit a quadratic polynomial to the walking points in the speed domain
coef_fr_walk = #TODO: Fit a quadratic polynomial to the walking points in the Froude number domain

# Extend walking fit domain to include full speed range
x_speed_fit = #TODO: Create a speed array for fitting that covers the full range of speeds (np.linspace() can be used for this)
x_fr_fit = #TODO: Create a Froude number array for fitting that covers the full range of Froude numbers (np.linspace() can be used for this)

#TODO: Compute fitted values for walking fits (Hint: you can use np.polyval() to compute the fitted values from the coefficients and fit arrays)
y_speed_fit_walk = #TODO
y_fr_fit_walk = #TODO

#TODO: Do the same fitting process for the running points, but use a linear fit instead of a quadratic fit (Hint: set the degree to 1 in np.polyfit())
coef_speed_run = #TODO
coef_fr_run = #TODO

x_speed_fit_run = #TODO
x_fr_fit_run = #TODO

y_speed_fit_run = #TODO
y_fr_fit_run = #TODO

# Plot 1: Net CoT vs Speed
axs[0].plot(x_speed_fit, y_speed_fit_walk,
            label="Quadratic fit (walking)", linewidth=2)
axs[0].plot(x_speed_fit_run, y_speed_fit_run, "--",
            label="Linear fit (running)", linewidth=2)
axs[0].scatter(x_speed_walk, y_walk, marker="o", label="Walking points")
axs[0].scatter(x_speed_run, y_run, marker="s", s=70,
               label="Running points", zorder=3)
for index, label in enumerate(stage_labels):
    axs[0].annotate(label, (speeds_mps[index], net_cot_j_per_kg_m[index]),
                    textcoords="offset points", xytext=(5, 5), fontsize=8)
axs[0].set_xlabel("Speed (m/s)")
axs[0].set_ylabel("Net CoT (J·kg⁻¹·m⁻¹)")
axs[0].set_title("Net CoT vs Speed")
axs[0].set_ylim([0, 4.0])
axs[0].grid(True, alpha=0.3)
axs[0].legend()

# Plot 2: Net CoT vs Froude Number
axs[1].plot(x_fr_fit, y_fr_fit_walk,
            label="Quadratic fit (walking)", linewidth=2)
axs[1].plot(x_fr_fit_run, y_fr_fit_run, "--",
            label="Linear fit (running)", linewidth=2)
axs[1].scatter(x_fr_walk, y_walk, marker="o", label="Walking points")
axs[1].scatter(x_fr_run, y_run, marker="s", s=70,
               label="Running points", zorder=3)
for index, label in enumerate(stage_labels):
    axs[1].annotate(label, (fr_stage[index], net_cot_j_per_kg_m[index]),
                    textcoords="offset points", xytext=(5, 5), fontsize=8)
axs[1].axvline(0.25, linestyle="--", linewidth=1, label="Theory Fr=0.25")
axs[1].axvline(0.50, linestyle="--", linewidth=1, label="Theory Fr=0.50")
axs[1].set_xlabel("Froude Number")
axs[1].set_ylabel("Net CoT (J·kg⁻¹·m⁻¹)")
axs[1].set_title("Net CoT vs Froude Number")
axs[0].set_ylim([0, 4.0])
axs[1].grid(True, alpha=0.3)
axs[1].legend()

plt.tight_layout()
plt.show()

### III. Discussion Question Set 1

1. At what speed was your interpolated net CoT lowest? Was your interpolated value higher or lower than the prediction? What was the absolute difference in values? (Just try to estimate visually)
2. At what Froude number was your interpolated net CoT lowest? Was it higher or lower than your assumed optimal value? What was the absolute difference in values? (Just try to estimate visually)
3. Does the data suggest a transition region from walking to running near the predicted transition Froude number? Explain using your plot trends.
4. Is this data consistent with the inverted pendulum model regarding cost of transport? Give at least one reason why it is and why it isn't. (Remember that the inverted pendulum model is only for walking)
5. Refer back to the plot from Scientific American in the background section. Why do you think a human on a bicycle is so efficient compared to walking or to any of the other data shown?

### Answer Here

## IV. HR/HRR and VO₂ Relationship

### Background Theory

During steady-state submaximal exercise, oxygen uptake ($\dot{VO}_2$) and cardiovascular demand tend to rise together in a roughly linear way. This makes heart rate (HR) a convenient **field-ready proxy for metabolic intensity**, since HR is easy to monitor with wearables and chest straps.

However, HR depends on both work rate and the current state of the body (hydration, temperature, fatigue, caffeine, etc.), so we usually work with a normalized metric.

#### Heart Rate Reserve (HRR)

**Heart rate reserve (HRR)** quantifies how far the current heart rate is above resting, relative to the individual's maximal capacity:

$$HRR = HR - HR_{rest}$$

A common way to express it is as a percentage:

$$\%HRR = \frac{HR - HR_{rest}}{HR_{peak} - HR_{rest}} \times 100$$

This is useful because **%HRR tends to track %VO₂ reserve** (i.e., how far the current VO₂ is above resting and below maximal), especially during steady-state submaximal effort.

#### Why HRR is helpful

- **HR is accessible:** Most people can measure HR outside a lab, while VO₂ requires specialized equipment.
- **Relative scaling:** HRR accounts for individual differences in resting and maximal heart rate, making comparisons between people more meaningful.
- **Training zones:** Many exercise prescriptions use %HRR (or %VO₂ reserve) to define intensity zones (e.g., 60–70% HRR = moderate intensity).

**Keep in mind:** The HR–VO₂ relationship is not perfectly linear at very low or very high intensities, and it can shift during long-duration exercise due to cardiac drift (HR rising even when work rate is constant). Factors like dehydration, heat, caffeine, and medications can also alter heart rate independently of VO₂.

In [ ]:
# Stage-level HR and VO2 relationships (last 30 s of each stage)
stage_order = [
    "Rest", *stage_labels] if "Rest" in vo2_data["Stage"].dropna().unique() else stage_labels.copy()
stage_summary = []

for stage in stage_order:
    stage_data = vo2_data[vo2_data["Stage"] == stage]
    if stage_data.empty:
        continue
    last_30s = stage_data[stage_data["Time[s]"]
                          >= (stage_data["Time[s]"].max() - 30)]
    stage_summary.append({
        "Stage": stage,
        "VO2[mL/min]": last_30s["VO2[mL/min]"].mean(),
        "HR[bpm]": last_30s["HR[bpm]"].mean()
    })

hr_vo2_df = pd.DataFrame(stage_summary)

# Compute HRR from resting stage
if "Rest" in hr_vo2_df["Stage"].values:
    hr_rest = hr_vo2_df.loc[hr_vo2_df["Stage"] == "Rest", "HR[bpm]"].iloc[0]
else:
    hr_rest = hr_vo2_df["HR[bpm]"].min()
    print("Warning: No explicit Rest stage found. Using minimum stage HR as resting estimate.")

hr_peak = 220 - 27  # Age-predicted max HR (220 - age)
hr_vo2_df["HRR[bpm]"] = hr_vo2_df["HR[bpm]"] - hr_rest
hr_vo2_df["%HRR"] = 100 * (hr_vo2_df["HR[bpm]"] -
                           hr_rest) / (hr_peak - hr_rest)

# Linear fits
m_hr, b_hr = np.polyfit(hr_vo2_df["HR[bpm]"], hr_vo2_df["VO2[mL/min]"], 1)
m_hrr, b_hrr = np.polyfit(hr_vo2_df["HRR[bpm]"], hr_vo2_df["VO2[mL/min]"], 1)

# Coefficient of determination (R^2)
r2_hr_vo2 = np.corrcoef(hr_vo2_df["HR[bpm]"],
                        hr_vo2_df["VO2[mL/min]"])[0, 1] ** 2
r2_hrr_vo2 = np.corrcoef(
    hr_vo2_df["HRR[bpm]"], hr_vo2_df["VO2[mL/min]"])[0, 1] ** 2

fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: HR vs VO2
x1 = hr_vo2_df["HR[bpm]"].to_numpy()
y = hr_vo2_df["VO2[mL/min]"].to_numpy()
axs[0].scatter(x1, y, s=70)
x1_fit = np.linspace(x1.min(), x1.max(), 100)
axs[0].plot(x1_fit, m_hr * x1_fit + b_hr, linewidth=2)
for _, row in hr_vo2_df.iterrows():
    axs[0].annotate(row["Stage"], (row["HR[bpm]"], row["VO2[mL/min]"]),
                    textcoords="offset points", xytext=(6, 6), fontsize=9)
axs[0].set_xlabel("HR (bpm)")
axs[0].set_ylabel("VO2 (mL/min)")
axs[0].set_title(f"HR vs VO2 (R² = {r2_hr_vo2:.3f})")
axs[0].grid(True, alpha=0.3)

# Plot 2: HRR vs VO2
x2 = hr_vo2_df["HRR[bpm]"].to_numpy()
axs[1].scatter(x2, y, s=70)
x2_fit = np.linspace(x2.min(), x2.max(), 100)
axs[1].plot(x2_fit, m_hrr * x2_fit + b_hrr, linewidth=2)
for _, row in hr_vo2_df.iterrows():
    axs[1].annotate(row["Stage"], (row["HRR[bpm]"], row["VO2[mL/min]"]),
                    textcoords="offset points", xytext=(6, 6), fontsize=9)
axs[1].set_xlabel("HRR (bpm)")
axs[1].set_ylabel("VO2 (mL/min)")
axs[1].set_title(f"HRR vs VO2 (R² = {r2_hrr_vo2:.3f})")
axs[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Resting HR used for HRR: {hr_rest:.1f} bpm")
print(f"Peak HR in trial: {hr_peak:.1f} bpm")
print(f"R² for HR vs VO2:  {r2_hr_vo2:.3f}")
print(f"R² for HRR vs VO2: {r2_hrr_vo2:.3f}")

hr_vo2_df

### Discussion Question Set 2

1. If you measured this participant's heart rate to be 106 beats per minute, what would their %HRR value be if their resting heart rate was 60 beats per minute and they were 21 years old (use $HR_{max}=220$ - age)?
2. Based on the HRR plot above, what is the participant's estimated $\dot{V}_{O_{2}}$ if their HRR is 35 beats per minute?
3. Are there any points in the plots above that don't follow the expected trend? Why do you think this might be? (Hint: read the background section)

Answer Here

# V. Notebook Export

In [ ]:
import nbformat
from nbconvert import HTMLExporter
from google.colab import files

# Helper function to export a Colab notebook as an HTML file
def export_current_notebook(notebook_path, output_filename="exported_notebook.html"):
    """
    Exports the specified Colab notebook (with all cell outputs) as an HTML file.
    """
    try:
        # Convert to HTML
        with open(notebook_path, "r", encoding="utf-8") as f:
            notebook_content = nbformat.read(f, as_version=4)

        # Convert notebook to HTML
        html_exporter = HTMLExporter()
        html_exporter.exclude_input = False
        html_exporter.exclude_output = False
        html_data, _ = html_exporter.from_notebook_node(notebook_content)

        # Add CSS to wrap lines and prevent clipping
        wrap_css = """
        <style>
            pre, code {
                word-wrap: break-word;
                white-space: pre-wrap;
                word-break: break-word;
            }
        </style>
        """
        html_data = wrap_css + html_data

        # Save the HTML file
        with open(output_filename, "w", encoding="utf-8") as f:
            f.write(html_data)

        print(f"Notebook exported successfully as {output_filename}")

        # Optionally download the file
        files.download(output_filename)

    except Exception as e:
        print(f"An error occurred: {e}")


# Change to your notebook path
notebook_to_export = #TODO: Copy the path to this notebook from the file tree and paste it here as a string
# This will save to your local downloads
output_filename = 'cost_of_transport_lab.html'

export_current_notebook(notebook_path=notebook_to_export,
                        output_filename=output_filename)